# 🌾 Project 27 — Detect Crop Growth

## Goal

Track crop development during one growing season using:

- Monthly satellite composites
- NDVI calculation
- Time-series analysis
- Charts

---

# Why is this useful in Remote Sensing?

Crop monitoring is one of the most important applications of satellite imagery.

This workflow is used for:

- 🌱 Crop health monitoring
- 🌾 Yield prediction
- 💧 Irrigation management
- 🌍 Drought assessment
- 🚜 Precision agriculture

Farmers and researchers want to know:

- When did crops start growing?
- When was vegetation at maximum growth?
- When did crops mature or decline?

---

# New Skill: Time-Series Analysis

Previously:

```text
One Date
   ↓
NDVI
   ↓
Vegetation Map
````

Now:

```text
January Image
       ↓
     NDVI

February Image
       ↓
     NDVI

March Image
       ↓
     NDVI

       ↓

NDVI Time Series

       ↓

Growth Pattern
```

---

# Connect to Previous Lessons

| Project | Skill                  |
| ------- | ---------------------- |
| 6       | Calculate NDVI         |
| 16      | NDVI Time Series       |
| 17      | Monthly Greenest Image |
| 22      | Median Composite       |
| **27**  | Crop growth monitoring |

You already know all individual pieces.

Now we combine them into a real monitoring workflow.

---

# The Big Idea

A crop does not stay the same during the year.

Example:

```text
NDVI

1.0 |
    |
0.8 |              🌿🌿
    |           🌿      🌿
0.5 |       🌿
    |
0.2 | 🌱
    |
0.0 |________________________

     Jan Feb Mar Apr May Jun
```

The curve tells us the crop lifecycle.

---

# Workflow

```text
Sentinel-2 Images
        ↓
Filter Growing Season
        ↓
Create Monthly Composite
        ↓
Calculate NDVI
        ↓
Extract Average NDVI
        ↓
Create Chart
        ↓
Analyze Crop Growth
```

---

# Step 1 — Define Crop Area

We'll create a small agricultural area.

```javascript
var cropArea = ee.Geometry.Polygon([
  [
    [13.30,52.30],
    [13.50,52.30],
    [13.50,52.45],
    [13.30,52.45],
    [13.30,52.30]
  ]
]);
```

This represents our field region.

---

# Step 2 — Load Sentinel-2 Collection

```javascript
var collection = ee.ImageCollection(
  "COPERNICUS/S2_SR_HARMONIZED"
)
.filterBounds(cropArea)
.filterDate(
  '2024-03-01',
  '2024-09-30'
);
```

Why March to September?

Because this represents a typical growing season.

Workflow:

```text
ImageCollection

March → September

Many satellite images
```

---

# Step 3 — Create Monthly Composites ⭐

This is today's main concept.

Instead of:

```javascript
.first()
```

we create one image per month.

---

We need a function.

```javascript
function monthlyNDVI(month){

}
```

The function will:

1. Select one month
2. Create a composite
3. Calculate NDVI

---

# Step 4 — Create Monthly NDVI Function

```javascript
function monthlyNDVI(month){

  var start = ee.Date.fromYMD(
    2024,
    month,
    1
  );

  var end = start.advance(
    1,
    'month'
  );


  var image = collection
    .filterDate(start,end)
    .median();


  var ndvi = image
    .normalizedDifference([
      'B8',
      'B4'
    ])
    .rename('NDVI');


  return ndvi
    .set(
      'month',
      month
    );
}
```

---

# Understanding the Function

## Create Start Date

```javascript
ee.Date.fromYMD(2024,month,1)
```

Example:

If:

```text
month = 5
```

Earth Engine creates:

```text
2024-05-01
```

---

## Create End Date

```javascript
start.advance(1,'month')
```

Example:

```text
May 1

+

1 month

=

June 1
```

So we analyze:

```text
May 1 → June 1
```

---

## Monthly Composite

```javascript
collection
.filterDate(start,end)
.median()
```

Meaning:

```text
All May images

      ↓

Median Composite

      ↓

One May image
```

---

## Calculate NDVI

Same formula you already know:

```text
NDVI = (NIR - Red)
       ------------
       (NIR + Red)
```

Sentinel-2:

```text
B8 = NIR

B4 = Red
```

---

# Step 5 — Create Months List

```javascript
var months = ee.List.sequence(
  3,
  9
);
```

This creates:

```text
[3,4,5,6,7,8,9]
```

Meaning:

```text
March
April
May
June
July
August
September
```

---

# Step 6 — Apply Function to Every Month

```javascript
var monthlyImages = ee.ImageCollection(
  months.map(monthlyNDVI)
);
```

Important concept:

## `.map()`

means:

```text
Month 3
 ↓
monthlyNDVI()

Month 4
 ↓
monthlyNDVI()

Month 5
 ↓
monthlyNDVI()
```

It repeats the function automatically.

---

# Step 7 — Create a Chart

Now we want to know:

> How did average crop NDVI change?

We use:

```javascript
ui.Chart.image.series()
```

---

```javascript
var chart = ui.Chart.image.series({
  imageCollection: monthlyImages,
  region: cropArea,
  reducer: ee.Reducer.mean(),
  scale: 10
})
.setOptions({
  title:'Crop Growth NDVI',
  vAxis:{
    title:'NDVI'
  },
  hAxis:{
    title:'Month'
  }
});

print(chart);
```

---

# What Does the Chart Show?

Example:

```text
NDVI

0.8 |
    |          /\
0.6 |        /    \
    |      /
0.4 |    /
    |
0.2 |
    |________________

    Mar Apr May Jun Jul Aug Sep
```

Interpretation:

## Early season:

```text
Low NDVI
```

↓

## Crop growth:

```text
Increasing NDVI
```

↓

## Peak vegetation:

```text
Maximum NDVI
```

↓

## Harvest/senescence:

```text
Declining NDVI
```

---

# Complete Code

```javascript
// 1. Crop Area
var cropArea = ee.Geometry.Polygon([
  [
    [13.30,52.30],
    [13.50,52.30],
    [13.50,52.45],
    [13.30,52.45],
    [13.30,52.30]
  ]
]);


// 2. Sentinel-2 Collection
var collection = ee.ImageCollection(
  "COPERNICUS/S2_SR_HARMONIZED"
)
.filterBounds(cropArea)
.filterDate(
  '2024-03-01',
  '2024-09-30'
);


// 3. Monthly NDVI Function
function monthlyNDVI(month){

  var start = ee.Date.fromYMD(
    2024,
    month,
    1
  );

  var end = start.advance(
    1,
    'month'
  );


  var image = collection
    .filterDate(start,end)
    .median();


  var ndvi = image
    .normalizedDifference([
      'B8',
      'B4'
    ])
    .rename('NDVI');


  return ndvi.set(
    'month',
    month
  );
}


// 4. Months
var months = ee.List.sequence(
  3,
  9
);


// 5. Monthly Collection
var monthlyImages = ee.ImageCollection(
  months.map(monthlyNDVI)
);


// 6. Chart
var chart = ui.Chart.image.series({
  imageCollection: monthlyImages,
  region: cropArea,
  reducer: ee.Reducer.mean(),
  scale:10
})
.setOptions({
  title:'Crop Growth NDVI',
  vAxis:{
    title:'NDVI'
  }
});


print(chart);
```

---

# Things to Remember

## 1. Time Series

A time series means:

```text
Same location

Different dates

↓

Observe change over time
```

---

## 2. Monthly Composite

Instead of using one image:

```text
May Image 1
May Image 2
May Image 3

↓

median()

↓

May Composite
```

Benefits:

* Less cloud
* More stable NDVI
* Better comparison

---

## 3. `.map()`

Very important GEE concept.

It means:

```text
Apply the same function
to every item in a collection
```

Example:

```javascript
months.map(monthlyNDVI)
```

means:

```text
Run monthlyNDVI()
for every month
```

---

# Common Beginner Mistakes

## ❌ Mistake 1: Comparing NDVI from random dates

Wrong:

```text
March 5
July 20
September 10
```

Better:

```text
Monthly composites
```

because each month represents a consistent period.

---

## ❌ Mistake 2: Using one cloudy image

Clouds can create artificial NDVI changes.

Better:

```text
Monthly median composite
```

---

## ❌ Mistake 3: Forgetting that crops have seasons

High NDVI does not always mean "better."

Example:

```text
After harvest:

NDVI decreases naturally.
```

This is a crop lifecycle, not necessarily damage.

---

# Your Practice (Project 27.1)

Create a crop growth monitoring workflow.

## Requirements

1. Choose an agricultural region.
2. Load Sentinel-2 imagery.
3. Select a growing season.
4. Create monthly median composites.
5. Calculate monthly NDVI.
6. Create an NDVI time-series chart.
7. Explain:

   * When vegetation starts increasing.
   * When peak growth occurs.
   * When vegetation declines.

---

# Conceptual Questions

## Question 1

Why do we create **monthly composites** instead of calculating NDVI from one image per month?

---

## Question 2

What does this line do?

```javascript
months.map(monthlyNDVI)
```

Why is `.map()` useful in Earth Engine?

---

## Question 3

If NDVI decreases after July, does that always mean crops are damaged? Explain why or why not.

```
```
